## Building a product that converts Python code to C++ for performance using open-source models

##### There are many open source models that are fine tuned only for coding purposes. However I will be using the normal qwen3:30b,  llama3.1:8b and qwen2.5 coder:7b and comapring the output. However these are not the correct models for this task beacuse specialised models exist for coding like qwen2.5 Coder, DeepSeek Coder V2 Lite, gpt-oss-20b, etc.

##### Paid closed-source models, like those from OpenAI or Anthropic, are typically accessed via API keys retrieved securely from environment variables (e.g., os.getenv('CLIENT_API_KEY')). This approach keeps keys out of codebases, supports serverless scaling, and enables easy integration with their official SDKs (e.g., OpenAI's Python client).

##### Open-source models, by contrast, are downloaded and run locally using tools like Ollama, which simplifies serving them on your machine or server. Once installed (e.g., ollama pull llama3), you interact via Ollama's API client, OpenAI-compatible clients for broad compatibility, or model-specific libraries if available. Key benefits include zero inference costs, full customization (e.g., fine-tuning with LoRA), and privacy since data stays local—ideal for LLM engineering workflows.

#### Ways to improve workflows with llms:-
    - Quantization
    - API Caching and Retries
    - Cost Monitoring for Closed Models

In [35]:

from openai import OpenAI
import gradio as gr
import subprocess

In [36]:
# Connect to Client Library

ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [37]:
models = ["qwen2.5-coder:latest", "qwen3:30b", "llama3.1:8b"] # list of model names to be used in the dropdown menu

clients = {"qwen2.5-coder:latest":ollama, "qwen3:30b":ollama, "llama3.1:8b":ollama} # mapping from model name to client instance

In [38]:
## To run c++ code first we need to make sure if the system has necessary compiler installed and then we can use subprocess module to run the c++ code and get the output.

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '13th Gen Intel(R) Core(TM) i7-13650HX',
  'cores_logical': 20,
  'cores_physical': 14,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (Rev13, Built by MSYS2 project) 15.2.0',
   'g++': 'g++.EXE (Rev13, Built by MSYS2 project) 15.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [39]:
message = """You are an expert in C++ development environments. 
Analyze the following system information and tell me exactly which commands to run (using available package managers like winget) to 
install everything necessary to compile and run C++ code on this machine. 
Provide step-by-step installation commands and a simple test compile command.
Look at the system information and generate the response in the below format
Remember that from the system information you can determine which package manager is available and which compiler is already installed, so only include the necessary installation commands.
Also you can use the entire resources of the system information to determine the best way to compile in least amount of time and steps.

Response Format:\n
## Required Installations
1. Command 1 explanation...
2. Command 2 explanation...
3. compile_command: The command to compile a simple C++ program
4. run_command: The command to run the compiled C++ Program.

## Verification Test
```cpp
// hello.cpp
```

Compile and run:
```cmd
> compile_command
> run_command
```

System Information:\n{system_info}
"""

In [40]:
response = ollama.chat.completions.create(
    model="qwen2.5-coder:latest",
    messages=[{"role":"user","content":message}]
)

In [41]:
from IPython.display import Markdown

Markdown(response.choices[0].message.content)

## Required Installations
1. **Ensure Package Manager is Up-to-Date**
   - For example, if `winget` (from Windows Store) is the package manager available:
     ```cmd
     winget upgrade --id Microsoft.Winget.Source
     ```
   
2. **Install C++ Build Tools**
   - Install necessary components for C++ development using Visual Studio's built-in tools which provide a complete compiler and toolchain.
     ```cmd
     winget install -i Microsoft.VisualStudio.Component.VC.Tools.x86.x64.ClangCXX --version X.X.X
     ```
   Replace `X.X.X` with the latest stable version.

## Verification Test

For a quick proof of setup, create a simple "Hello World" C++ program:

```cpp
// hello.cpp
#include <iostream>

int main() {
    std::cout << "Hello, World!" << std::endl;
    return 0;
}
```

Compile and run:
```cmd
> g++ -o hello hello.cpp  // Ensure 'g++' is recognized from the installation.
> hello
```

This command assumes `winget` installed a comprehensive C++ toolchain that includes both a compiler like GCC (`g++`) and other necessary elements for building applications. The final step tests if the environment is ready to compile and run simple C++ code effectively. Adjust the commands if winget installs tools with different names or flags.

#### Instead of installing what we can do is use the website 
https://www.programmiz.com/cpp-programming/online-compiler/
### to test our generated code.

### Lets complete the main task for now

In [42]:
system_prompt = """
Your task is to convert Python Code into high performance C++ code.
Respond only with C++ Code. Do not provide any explanation other than occasional comments.
The c++ response need to produce and identical output in the fasted possible time.
"""

def user_prompt(python):
    return f"""
    Port the following python code into C++ with the fastest possible implementation that produces the same output in the least amount of time.
    The system information is as follows:\n{system_info}\n\n
    Your response will be written to a file called main.cpp and then it will be compiled and executed.
    
    Python Code to port:\n
    ```python
    {python}
    ```
    """

In [43]:
def messagesFor(python):
    return [
        {"role":"system","content":system_prompt},
        {"role":"user", "content":user_prompt(python)}
    ]

In [44]:
def writeOutput(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [45]:
def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(
        model=model,
        messages=messagesFor(python)
    )
    reply = response.choices[0].message.content
    reply = reply.replace("```cpp", "").replace("```","")
    writeOutput(reply)
    return reply
    

In [58]:
pyCode = """
import time

def calculate(iteration,param1,param2):
    result = 1.0
    for i in range(1,iteration):
        j = i * param1 / param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

startTime = time.time()
result = calculate(200000000,4,1) * 4
endTime = time.time()

print(f"Result: {result}")
print(f"Execution Time: {endTime - startTime}")

"""

In [59]:
# port("qwen2.5-coder:latest", pyCode)

##### This function takes a string of Python code, executes it using exec() inside a controlled global environment, and temporarily redirects standard output to a buffer so that anything printed by the code can be captured instead of displayed on the console. If the code runs successfully, the printed output is stored; if an error occurs, the exception message is captured instead. Finally, the original output stream is restored so the rest of the program behaves normally (though the function should include a return output to actually give back the result).

##### What do the varibales mean?
- globals_dict → a dictionary representing the global namespace for execution
    - “globals” = variables available at the top level
    - “dict” = it’s a Python dictionary
- __builtins__ → a special Python name that gives access to built-in functions like print, len, etc.
- buffer → acts like a temporary storage area (an in-memory file) to collect output
- old_stdout → saves the original sys.stdout so it can be restored later
- sys.stdout → the standard output stream (where print() normally writes)
- 

In [60]:
import io, sys
def runPython(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {str(e)}"

    finally:
        sys.stdout = old_stdout

    return output

In [49]:
# import subprocess

def compileAndRun():
    try:
        subprocess.run(
            ["g++", "main.cpp", "-o", "main"],
            check=True, text=True, capture_output=True
        )

        print(subprocess.run(["main.exe"], check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(["main.exe"], check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(["main.exe"], check=True, text=True, capture_output=True).stdout)

    except subprocess.CalledProcessError as e:
        print(f"Compilation or execution failed: {e.stderr}")

In [50]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Text(label="Python code:", lines=28, value=pyCode)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [61]:
print(runPython(pyCode),"\n________________________________________________\n")
print(compileAndRun())


Result: 3.650237869724886
Execution Time: 19.317509174346924
 
________________________________________________

Result: 3.65024
Execution Time: 0.687298 seconds

Result: 3.65024
Execution Time: 0.701599 seconds

Result: 3.65024
Execution Time: 0.692855 seconds

None
